# LSTM Credit-Card Fraud Detection

This notebook trains an LSTM independently from the CNN notebook. The same train, validation, and test split is used so that the models can be compared fairly.

The dataset is tabular rather than naturally sequential. Therefore, the 30 input columns are represented as an artificial feature sequence with shape `(samples, 30, 1)`. The 30 columns are not real time steps.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

## Load Shared Preprocessed Data

In [ ]:
# Google Colab path used by the preprocessing notebook.
BASE_PATH = '/content/drive/MyDrive/Credit-Card-Fraud-Detection/'

# Mount Google Drive when this notebook is running in Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    # For local execution, set BASE_PATH to the project directory.
    BASE_PATH = os.path.abspath('../..') + os.sep

DATA_PATH = os.path.join(BASE_PATH, 'processed_data')
X_train = pd.read_csv(os.path.join(DATA_PATH, 'X_train.csv'))
X_val = pd.read_csv(os.path.join(DATA_PATH, 'X_val.csv'))
X_test = pd.read_csv(os.path.join(DATA_PATH, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(DATA_PATH, 'y_train.csv')).values.ravel()
y_val = pd.read_csv(os.path.join(DATA_PATH, 'y_val.csv')).values.ravel()
y_test = pd.read_csv(os.path.join(DATA_PATH, 'y_test.csv')).values.ravel()

print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)

## Reshape Data for LSTM

Keras LSTM input has the form `(samples, timesteps, features)`. Here, each transaction has 30 artificial feature steps and one value at each step.

In [ ]:
n_features = X_train.shape[1]
X_train_lstm = X_train.to_numpy(dtype=np.float32).reshape(-1, n_features, 1)
X_val_lstm = X_val.to_numpy(dtype=np.float32).reshape(-1, n_features, 1)
X_test_lstm = X_test.to_numpy(dtype=np.float32).reshape(-1, n_features, 1)

print('LSTM train shape:', X_train_lstm.shape)
print('LSTM validation shape:', X_val_lstm.shape)
print('LSTM test shape:', X_test_lstm.shape)

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print('Class weights:', class_weight)

## Build and Train the LSTM

In [ ]:
model = Sequential([
    LSTM(64, input_shape=(n_features, 1)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)
model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=30,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

## Training Curves

In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

history_df[['loss', 'val_loss']].plot(ax=axes[0], title='Loss')
history_df[['auc', 'val_auc']].plot(ax=axes[1], title='AUC')

for axis in axes:
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate on the Test Set

In [ ]:
test_metrics = model.evaluate(X_test_lstm, y_test, verbose=0, return_dict=True)
print('Test metrics:')
for name, value in test_metrics.items():
    print(f'{name}: {value:.4f}')

y_probability = model.predict(X_test_lstm, verbose=0).ravel()
y_prediction = (y_probability >= 0.5).astype(int)

print('Test ROC-AUC:', f'{roc_auc_score(y_test, y_probability):.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_prediction, target_names=['Legitimate', 'Fraud'], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_prediction)
ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud']).plot(cmap='Blues')
plt.title('LSTM Confusion Matrix')
plt.show()

## Save the Best LSTM Model

In [ ]:
MODEL_DIR = os.path.join(BASE_PATH, 'models', 'LSTM')
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'LSTM_best_model.keras')
model.save(MODEL_PATH)
print('Model saved to:', MODEL_PATH)